In [1]:
import pandas as pd
import sqlite3

## Подробно:

1. Создай соединение с базой данных с помощью библиотеки sqlite3.
2. Одним запросом на группу создай два DataFrame: test_results и control_results - с колонками "time" и "avg_diff", и ровно двумя строками.
    - В "time" должны быть значения "after" и "before".
    - В "avg_diff" - средняя дельта для всех пользователей за периоды до и после их первого визита на страницу.
    - Учитывай только тех пользователей, у кого есть наблюдения и «до», и «после».
3. Лабораторную project1 по-прежнему не учитывай.
4. Закрой соединение.
5. Дай ответ: подтвердилась ли гипотеза - влияет ли страница на поведение студентов?

In [2]:
db_path = '../data/checking-logs.sqlite'
conn = sqlite3.connect(db_path)

In [3]:
test_results = pd.io.sql.read_sql(
    """
    WITH labeled AS (
        SELECT
            t.uid,
            CASE WHEN UNIXEPOCH(t.first_commit_ts) < UNIXEPOCH(t.first_view_ts)
                 THEN 'before' ELSE 'after'
            END AS time,
            CAST((UNIXEPOCH(t.first_commit_ts) - d.deadlines) / 3600 AS INTEGER) AS diff
        FROM test t
        JOIN deadlines d ON t.labname = d.labs
        WHERE t.labname != 'project1' AND t.uid IS NOT NULL
    ),
    users_with_both AS (
        SELECT uid
        FROM labeled
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2
    ),
    user_avg AS (
        SELECT
            uid,
            time,
            AVG(diff) AS user_avg_diff
        FROM labeled
        WHERE uid IN (SELECT uid FROM users_with_both)
        GROUP BY uid, time
    )
    SELECT
        time,
        AVG(user_avg_diff) AS avg_diff
    FROM user_avg
    GROUP BY time;
    """,
    conn
)

print('=== Датафрейм test_results===')
test_results

=== Датафрейм test_results===


,time,avg_diff
0,after,-99.523810
1,before,-66.047619


In [4]:
control_results = pd.io.sql.read_sql(
    """
    WITH avg_view AS (
        SELECT AVG(UNIXEPOCH(first_view_ts)) AS avg_view_ts
        FROM test
    ),
    labeled AS (
        SELECT
            c.uid,
            CASE WHEN UNIXEPOCH(c.first_commit_ts) < (SELECT avg_view_ts FROM avg_view)
                 THEN 'before' ELSE 'after'
            END AS time,
            CAST((UNIXEPOCH(c.first_commit_ts) - d.deadlines) / 3600 AS INTEGER) AS diff
        FROM control c
        JOIN deadlines d ON c.labname = d.labs
        WHERE c.labname != 'project1' AND c.uid IS NOT NULL
    ),
    users_with_both AS (
        SELECT uid
        FROM labeled
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2
    ),
    user_avg AS (
        SELECT
            uid,
            time,
            AVG(diff) AS user_avg_diff
        FROM labeled
        WHERE uid IN (SELECT uid FROM users_with_both)
        GROUP BY uid, time
    )
    SELECT
        time,
        AVG(user_avg_diff) AS avg_diff
    FROM user_avg
    GROUP BY time;
    """,
    conn
)
print('===Датафрейм control_results===')
control_results

===Датафрейм control_results===


,time,avg_diff
0,after,-99.322222
1,before,-98.033333


In [5]:
conn.close()

$H_0$: появление "Ленты" не влияет на время начала выполнения заданий лабораторных работ. Средняя разница между первым коммитом и дедлайном до первого посещения страницы и после него одинакова.

- __A__ - группа *test*
- __B__ - Группа *control*

__Группа test__

| Период | Средняя дельта (часы) |
|--------|---------------------:|
| before | -66.047619 |
| after | -99.523810 |

__Группа control__

| Период | Средняя дельта (часы) |
|--------|---------------------:|
| before | -98.033333 |
| after | -99.322222 |

В тестовой группе после первого посещения страницы средняя дельта уменьшилась с -66.04 до -99.52 часов. Дельта по модулю стала больше, а это значит, что первый коммит стал происходить дальше от дедлайна, то есть студенты начали работу раньше. В контрольной группе изменения несущественны. Можно сделать вывод, что добавление "Ленты" стимулировало студентов раньше начинать выполнение лабораторных работ, **гипотеза $H_0$ не подтвердилась**. 

Предполагаемые причины, по которым добавление "Ленты" могло иметь эффект:
1. Социальное давление. Видя, какое количество однокурсников начало работу над лабораторной, студент может почувствовать необходимость не отставать от группы. Это может мотивировать его приступить к выполнению задания раньше.
2. Повышение вовлечённости в образование. Появление новой функциональности могло повысить интерес студентов к платформе. Видя, что компания развивает продукт и добавляет новые возможности, пользователи могли чаще посещать платформу и уделять больше внимания обучению.
3. Напоминание. Наблюдая активность других студентов в «Ленте новостей», пользователь мог чаще вспоминать о предстоящих заданиях и дедлайнах, что способствовало более раннему началу работы над лабораторными.